# OAPARP25 debug notebook
Run cells in order to inspect raw data and debug the year/hour swap logic.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

INPUT_ROOT = Path("../PARP")
SHEET_NAME_25 = "OAPARP25"
SHEET_NAME_61B = "OAPARP61-B"

module_path = Path("../src/clean_oaparp25.py").resolve()
spec = importlib.util.spec_from_file_location("clean_oaparp25", module_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

files = sorted([*INPUT_ROOT.rglob("*.xls"), *INPUT_ROOT.rglob("*.xlsx")])
print("files", len(files))

target = None
for p in files:
    engine = "xlrd" if p.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(p, engine=engine) as xl:
        if SHEET_NAME_25 in xl.sheet_names and SHEET_NAME_61B in xl.sheet_names:
            target = p
            break

print("target", target)
target

In [ ]:
engine = "xlrd" if target and target.suffix.lower() == ".xls" else "openpyxl"
with pd.ExcelFile(target, engine=engine) as xl:
    print("sheets", xl.sheet_names)
    raw_25 = pd.read_excel(xl, sheet_name=SHEET_NAME_25, header=None, dtype=str)
    raw_61b = pd.read_excel(xl, sheet_name=SHEET_NAME_61B, header=None, dtype=str)

raw_25.head(12)

In [ ]:
raw_61b.head(12)

In [ ]:
data_25 = mod.parse_sheet(raw_25)
data_61b = mod.parse_sheet(raw_61b)

print("OAPARP25 columns", list(data_25.columns))
print("OAPARP61-B columns", list(data_61b.columns))
data_25.head()

In [ ]:
well_col_61b = mod.find_column(data_61b.columns, ["WELL"])
year_col_61b = mod.find_column(data_61b.columns, ["YR", "TEST YEAR", "YEAR"])
month_col_61b = mod.find_column(data_61b.columns, ["MTH", "TEST MONTH", "MONTH"])

print("61B well_col", well_col_61b)
print("61B year_col", year_col_61b)
print("61B month_col", month_col_61b)

years_61b = data_61b[year_col_61b].apply(mod.normalize_year_value) if year_col_61b else pd.Series()
fallback_year = mod.get_mode(years_61b)

fallback_month = None
if month_col_61b:
    months = pd.to_numeric(data_61b[month_col_61b].apply(mod.clean_date_part), errors="coerce").dropna()
    if not months.empty:
        fallback_month = int(months.value_counts().idxmax())

print("fallback_year", fallback_year)
print("fallback_month", fallback_month)

In [ ]:
well_col_25 = mod.find_column(data_25.columns, ["WELL"])
year_col_25 = mod.find_column(data_25.columns, ["TEST YEAR", "YR", "YEAR"])
month_col_25 = mod.find_column(data_25.columns, ["TEST MONTH", "MTH", "MONTH"])
hours_col_25 = mod.find_column(
    data_25.columns,
    [
        "TEST LENGTH IN HOURS",
        "TEST LENGTH IN HOUR",
        "TEST LENGTH HRS",
        "HRS",
    ],
)

print("25 well_col", well_col_25)
print("25 year_col", year_col_25)
print("25 month_col", month_col_25)
print("25 hours_col", hours_col_25)

hours_text = data_25[hours_col_25].astype(str).str.strip() if hours_col_25 else pd.Series()
hours_empty = data_25[hours_col_25].isna() | (hours_text == "") | (hours_text.str.lower() == "nan") if hours_col_25 else pd.Series()

print("rows with empty hours", int(hours_empty.sum()) if hours_col_25 else 0)

if hours_col_25 and well_col_25 and year_col_25:
    preview = data_25.loc[hours_empty, [well_col_25, year_col_25]]
    if month_col_25:
        preview = data_25.loc[hours_empty, [well_col_25, year_col_25, month_col_25]]
    display(preview.head(10))

In [ ]:
cleaned_one = mod.clean_sheet(target)
print("cleaned rows", len(cleaned_one))
cleaned_one.head()

In [ ]:
if hours_col_25 and year_col_25 and hours_col_25 in cleaned_one.columns and year_col_25 in cleaned_one.columns:
    empty_after = cleaned_one[cleaned_one[hours_col_25].isna()]
    print("empty hours after", len(empty_after))

cleaned_one[[c for c in ["WELL", year_col_25, hours_col_25, "TEST DATE"] if c in cleaned_one.columns]].head(10)